# N029 · 可变窗口与最短覆盖

**目标：** 写出窗口有效条件并按方向收缩。

**先修：** N020, N028。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 最长无重复；覆盖频次；缺失计数；收缩条件；可行性单调前提。

## 从问题到算法

可变窗口不是看到连续子串就能套。需要明确收缩何时恢复合法性。最长无重复子串在出现重复时不断移除左端；最短覆盖在满足目标计数时尽可能收缩并记录答案。两个问题的“何时更新答案”相反。最短覆盖必须计入目标重复字符，而非只计字符种类。

## 最小实现

**本章接口：** `length_of_longest_substring(s)`、`min_window(s, t)`

In [1]:
from collections import Counter

def length_of_longest_substring(s):
    counts=Counter(); left=best=0
    for right,c in enumerate(s):
        counts[c]+=1
        while counts[c]>1:
            counts[s[left]]-=1; left+=1
        best=max(best,right-left+1)
    return best

def min_window(s,t):
    if not t: return ''
    need=Counter(t); missing=len(t); left=0; best=None
    for right,c in enumerate(s):
        if need[c]>0: missing-=1
        need[c]-=1
        while missing==0:
            if best is None or right-left+1<best[1]-best[0]: best=(left,right+1)
            old=s[left]; need[old]+=1; left+=1
            if need[old]>0: missing+=1
    return '' if best is None else s[best[0]:best[1]]

## 正确性、前提与复杂度

每个右端点只向前走，左端点也不回退。无重复窗口收缩后是该右端点的最长合法后缀；覆盖窗口循环检查了该右端点所有可覆盖的左端位置，保留最短者。

**代价：** 平均 O(|s|+|t|) 时间、O(σ) 计数空间；保存最佳索引避免循环内复制答案。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='abcaac'; counts=Counter(); left=0; rows=[]
for right,c in enumerate(s):
    counts[c]+=1
    while counts[c]>1:
        counts[s[left]]-=1; left+=1
    rows.append((left,right,s[left:right+1],right-left+1))
show_table(['left','right','合法窗口','长度'],rows)

left,right,合法窗口,长度
0,0,a,1
0,1,ab,2
0,2,abc,3
1,3,bca,3
4,4,a,1
4,5,ac,2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert length_of_longest_substring('abcabcbb')==3
assert length_of_longest_substring('')==0
assert min_window('ADOBECODEBANC','ABC')=='BANC'
assert min_window('a','aa')==''
from itertools import product
for n in range(6):
    for s in map(''.join,product('ab',repeat=n)):
        ref=max((r-l for l in range(n+1) for r in range(l,n+1) if len(set(s[l:r]))==r-l),default=0)
        assert length_of_longest_substring(s)==ref
        for t in ['a','ab','aa','bba']:
            candidates=[s[l:r] for l in range(n) for r in range(l+1,n+1) if not (Counter(t)-Counter(s[l:r]))]
            expected=min(candidates,key=len) if candidates else ''
            ans=min_window(s,t)
            assert len(ans)==len(expected)
            if ans: assert not (Counter(t)-Counter(ans))
print('N029: 所有本章断言通过')

N029: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造只用集合无法处理重复目标字符的反例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释非负和窗口的适用条件。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 3. Longest Substring Without Repeating Characters（canonical）：[官方入口](https://leetcode.com/problems/longest-substring-without-repeating-characters/)
- 76. Minimum Window Substring（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。